# 00 - Setup: verify your environment

Run every cell top to bottom. Each section ends with a green check.

Prerequisites: `.env` filled in (copy `.env.example`), the ADB wallet in `wallet/`, and - depending on `LLM_PROVIDER` - an API key or a running Ollama with the configured model pulled (`ollama pull qwen3`).

In [ ]:
from cityops_harness.checks import ok, check
from cityops_harness.config import load_settings

settings = load_settings()
ok(f"settings loaded - provider={settings.llm_provider}, "
   f"langfuse={settings.langfuse_mode}, dsn={settings.db_dsn}")

## 1 · Autonomous Database

In [ ]:
from cityops_harness.db import get_connection

conn = get_connection(settings)
with conn.cursor() as cur:
    cur.execute("SELECT banner_full FROM v$version")
    banner = cur.fetchone()[0]
ok(banner.splitlines()[0])

## 2 · In-database embeddings (ONNX)

Embeddings run **inside** the database - no API key, no text egress. The
cell below loads Oracle's `ALL_MINILM_L12_V2` model from the public
OML-Resources object-storage bucket if it is not already present.

In [ ]:
ONNX_URI = ("https://objectstorage.us-ashburn-1.oraclecloud.com"
            "/n/adwc4pm/b/OML-Resources/o/all_MiniLM_L12_v2.onnx")

with conn.cursor() as cur:
    cur.execute(
        "SELECT COUNT(*) FROM user_mining_models WHERE model_name = :m",
        m=settings.embed_model,
    )
    present = cur.fetchone()[0] > 0

if not present:
    with conn.cursor() as cur:
        cur.execute("""
            BEGIN
              DBMS_VECTOR.LOAD_ONNX_MODEL_CLOUD(
                model_name => :m,
                credential => NULL,
                uri        => :u,
                metadata   => JSON('{"function":"embedding",'
                                   || '"embeddingOutput":"embedding",'
                                   || '"input":{"input":["DATA"]}}')
              );
            END;""", m=settings.embed_model, u=ONNX_URI)
    conn.commit()

with conn.cursor() as cur:
    cur.execute(
        f"SELECT VECTOR_EMBEDDING({settings.embed_model} USING 'hello' AS DATA)"
        " FROM dual"
    )
    vec = cur.fetchone()[0]
check(vec is not None, f"{settings.embed_model} embeds in-database")

## 3 · LLM provider

One env var (`LLM_PROVIDER`) drives both LLM consumers: LangChain chat
calls and the Oracle Agent Memory SDK (which speaks LiteLLM).

In [ ]:
from cityops_harness.llm import chat_model, litellm_spec, sdk_llm

print("LiteLLM spec:", litellm_spec(settings))
_ = sdk_llm(settings)  # constructs the Agent Memory SDK adapter

model = chat_model(settings)
reply = model.invoke("Reply with exactly: pong")
check("pong" in reply.content.lower(),
      f"{settings.llm_provider} responded: {reply.content[:60]!r}")

## 4 · Langfuse (optional)

`LANGFUSE_MODE=cloud` needs keys in `.env`; `local` needs the compose
stack up (`docker compose -f docker-compose.langfuse.yml -f docker-compose.langfuse.override.yml up -d`); `off` skips this section.

In [ ]:
from cityops_harness.tracing import init_tracing

handler = init_tracing(settings)
if handler is None:
    ok("Langfuse off - skipping")
else:
    traced = chat_model(settings).invoke(
        "Say: traced", config={"callbacks": [handler]}
    )
    from langfuse import get_client
    get_client().flush()
    ok(f"trace sent to Langfuse ({settings.langfuse_mode}) - "
       "check the UI for a '00_setup' trace")

## 5 · Summary

In [ ]:
from cityops_harness.state import verify

for desc, passed in verify(conn, "00_setup"):
    check(passed, desc)
ok("environment ready - continue to 01_self_improving_copilot")